## NB4 — ROGUE Purity Audit per (sample, K, GEP)

**Pipeline:** NB1 (GSEA cell-type scoring) → NB2 (`per_gep_labels.tsv`) → NB4 (this notebook).

**What this notebook does:**
- Hard-assigns each cell to its argmax cNMF GEP, per K.
- Scores every (sample, K, GEP) group with ROGUE, on cNMF's own HVG substrate.
- Runs three checks, kept explicitly separate rather than folded into one recommendation:
  1. **Contamination diagnostic** — label-matched subset of a GEP vs. the whole GEP.
  2. **Split/merge signal** — whole-GEP ROGUE vs. the original-annotation baseline.
  3. **Null control** — whole-GEP ROGUE vs. random same-size groups, same gene space.

**Gene space:** everything below uses cNMF's fixed HVG panel. Absolute ROGUE values are not
comparable to NB3's full-transcriptome scores or the 0.9 threshold from Liu et al. (2020) —
only deltas *within* this notebook are meaningful.

**Reference:** Liu et al., *Nat Commun* (2020). https://github.com/PaulingLiu/ROGUE


In [22]:
import os
os.environ['R_HOME'] = '/Library/Frameworks/R.framework/Resources'
import glob
import re
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, numpy2ri
from rpy2.robjects.packages import importr

CONVERTER = ro.default_converter + pandas2ri.converter + numpy2ri.converter
importr('ROGUE')
importr('tibble')  # ROGUE's SE_fun calls tibble() internally without qualifying it


rpy2.robjects.packages.Package as a <module 'tibble'>

### 1. Config

In [23]:
# Paths
ADATA_PATH = "hnc_data/hnc_myeloid_2021.h5ad"
COUNTS_LAYER = None

METADATA_PATH = "hnc_data/hnc_myeloid_globalcluster4.csv"
BARCODE_COL = None
METADATA_SEP = ","
BASELINE_LABEL_COL = "global.cluster4"

# Same cNMF outputs as NB1
CNMF_DIR = "hnc_consensus_outputs_k3-35/combined_hnc_k_consensus_outputs"
CNMF_PREFIX = "combined_hnc_k"
DENSITY_THRESHOLD = "0_01"
K_MIN, K_MAX = 3, 35
N_HVG = 5000

# NB2 output — one label per (K, GEP)
LABELS_PATH = "hnc_combined5000hvg_clanp_results/k3-35_corr-tree_kl-divergence/per_gep_labels.tsv"
LABELS_SEP = "\t"
AMBIGUOUS_LABEL = "Ambiguous/Mixed"
DROP_AMBIGUOUS = False
FAMILY_LABEL_SUFFIX = "_family"

# Must match NB2's family definitions
CELLTYPE_FAMILIES = {
    "Monocyte": [
        "Mono_CD14", "Mono_CD14_ID1", "Mono_CD14_IL1B", "Mono_CD14_THBS1",
        "Mono_CD16", "Mono_Int", "Mono_TIL"
    ],
    "DC": [
        "cDC1_CLEC9A", "cDC2_CD1C", "cDC2_CD33", "mregDC_LAMP3"
    ],
    "Macrophage": [
        "Mac_CXCL9", "Mac_IL1B", "Mac_IL1Bint", "Mac_SPP1"
    ],
}

# ROGUE settings
ROGUE_PLATFORM = "UMI"
ROGUE_K = 45
MIN_CELLS_PER_GROUP = 15
DELTA_ROGUE_THRESHOLD = 0.05

# Null control settings
N_NULL_DRAWS = 10
NULL_SIZE_BIN = 25   # round group sizes to the nearest N cells before drawing a null
RANDOM_SEED = 0

OUT_DIR = "hnc_combined5000hvg_clanp_results/rogue_purity_results_per_gep"
os.makedirs(OUT_DIR, exist_ok=True)

print("Configuration loaded.")


Configuration loaded.


### 2. Load raw counts + metadata

In [24]:
adata = sc.read_h5ad(ADATA_PATH)
X = adata.layers[COUNTS_LAYER] if COUNTS_LAYER else adata.X
if sparse.issparse(X):
    X = X.toarray()
X = X.astype(np.float64)
if not np.allclose(X, np.round(X)):
    raise ValueError("Matrix doesn't look like raw counts — check COUNTS_LAYER.")

counts_df = pd.DataFrame(X.T, index=adata.var_names, columns=adata.obs_names)

meta = pd.read_csv(METADATA_PATH, sep=METADATA_SEP,
                   index_col=0 if BARCODE_COL is None else None)
if BARCODE_COL is not None:
    meta = meta.set_index(BARCODE_COL)
meta.index = meta.index.astype(str)

common = counts_df.columns.intersection(meta.index)
counts_df = counts_df[common]
meta = meta.loc[common]
meta['orig.ident'] = adata.obs.loc[meta.index, 'orig.ident'].astype(str)

samples = meta['orig.ident']
original_labels = meta[BASELINE_LABEL_COL].astype(str)

print(f"Aligned {len(common)} cells across {samples.nunique()} samples")
print(original_labels.value_counts().to_string())


Aligned 26444 cells across 63 samples
global.cluster4
Mono_CD14          7518
Mono_CD16          2503
Mac_IL1Bint        1842
Mono_CD14_IL1B     1801
Mono_CD14_THBS1    1783
Mono_Int           1684
cDC2_CD33          1319
Mac_CXCL9          1225
DC_pDC             1218
Mono_CD14_ID1      1154
Mono_TIL            988
cDC2_CD1C           966
Mac_IL1B            774
Mac_SPP1            573
mregDC_LAMP3        482
Mast                313
cDC1_CLEC9A         301


### 3. cNMF's HVG panel

- cNMF factorizes every K on one fixed gene panel, chosen once before K varies.
- Any `gene_spectra_score` file's columns are that panel — read one, verify the rest agree.
- `counts_df_hvg` is the gene space every ROGUE call below uses.


In [25]:
gene_spectra_pattern = os.path.join(
    CNMF_DIR, f"{CNMF_PREFIX}.gene_spectra_score.k_*.dt_{DENSITY_THRESHOLD}.txt"
)
gene_spectra_files = sorted(glob.glob(gene_spectra_pattern))
if not gene_spectra_files:
    raise FileNotFoundError(f"No gene_spectra_score files found matching {gene_spectra_pattern}")

hvg_genes = pd.read_csv(gene_spectra_files[0], sep='\t', index_col=0, nrows=0).columns.tolist()
for f in gene_spectra_files[1:]:
    other = pd.read_csv(f, sep='\t', index_col=0, nrows=0).columns.tolist()
    if other != hvg_genes:
        raise ValueError(f"Gene panel in {f} does not match {gene_spectra_files[0]}.")

missing_from_adata = [g for g in hvg_genes if g not in counts_df.index]
if missing_from_adata:
    raise ValueError(f"{len(missing_from_adata)} cNMF genes missing from AnnData, e.g. {missing_from_adata[:5]}")

counts_df_hvg = counts_df.loc[hvg_genes]
print(f"HVG panel: {len(hvg_genes)} genes (verified identical across {len(gene_spectra_files)} K values)")
print(f"HVG counts matrix: {counts_df_hvg.shape}")


HVG panel: 5000 genes (verified identical across 33 K values)
HVG counts matrix: (5000, 26444)


### 4. ROGUE scoring helpers

- `expr_full` is pushed to R once here and reused by every ROGUE call in this notebook.
- `rogue_for_cells` — ROGUE for one arbitrary set of cells.
- `rogue_for_groups` — ROGUE for every (sample, label) group in a labeled Series, within-sample.


In [26]:
ro.r("suppressMessages(library(ROGUE)); suppressMessages(library(tibble))")  # ROGUE's SE_fun calls tibble() unqualified — must be attached, not just imported
with CONVERTER.context():
    ro.globalenv['expr_full'] = counts_df_hvg
ro.r("expr_full <- as.matrix(expr_full)")

def rogue_for_cells(cell_ids):
    """ROGUE for one group of cells. Reuses expr_full already in the R session."""
    cell_ids = list(cell_ids)
    with CONVERTER.context():
        ro.globalenv['cells_keep'] = ro.StrVector(cell_ids)
    try:
        ro.r(f"""
            expr_mat  <- expr_full[, cells_keep, drop=FALSE]
            expr_mat  <- matr.filter(expr_mat, min.cells = 10, min.genes = 10)
            ent_res   <- SE_fun(expr_mat, span = 0.6, mt.method = "fdr")
            rogue_val <- CalculateRogue(ent_res, platform = "{ROGUE_PLATFORM}", k = {ROGUE_K})
        """)
        with CONVERTER.context():
            return float(ro.r('rogue_val')[0])
    except Exception:
        return float('nan')

def rogue_for_groups(labels, samples, min_cells=MIN_CELLS_PER_GROUP, desc="ROGUE"):
    """ROGUE per (sample, label), within-sample. Returns [sample, cluster, ROGUE, n_cells]."""
    counts = labels.value_counts()
    keep = counts[counts >= min_cells].index
    labels = labels[labels.isin(keep)]
    samples = samples.loc[labels.index]

    pairs = (pd.DataFrame({'sample': samples.values, 'cluster': labels.values})
               .value_counts().reset_index(name='n_cells'))
    pairs = pairs[pairs['n_cells'] >= min_cells]

    rows = []
    for row in tqdm(list(pairs.itertuples(index=False)), desc=desc):
        mask = (samples == row.sample) & (labels == row.cluster)
        rows.append({'sample': row.sample, 'cluster': row.cluster,
                     'ROGUE': rogue_for_cells(labels.index[mask]), 'n_cells': row.n_cells})
    return pd.DataFrame(rows).dropna(subset=['ROGUE']).reset_index(drop=True)


### 5. NB2 per-GEP labels

- Loads `per_gep_labels.tsv`: one label per (K, GEP).
- A label is either an original `global.cluster4` cell type, a `<family>_family` label, or `Ambiguous/Mixed`.
- Anything else is flagged — it would silently fail to match a baseline later.


In [27]:
labels_df = pd.read_csv(LABELS_PATH, sep=LABELS_SEP)
required_cols = {"k", "gep", "label"}
missing = required_cols - set(labels_df.columns)
if missing:
    raise ValueError(f"per-GEP labels TSV missing columns {missing}. Got: {list(labels_df.columns)}")

labels_df['k']     = labels_df['k'].astype(int)
labels_df['gep']   = labels_df['gep'].astype(str)
labels_df['label'] = labels_df['label'].astype(str)
label_lookup = labels_df.set_index(['k', 'gep'])['label'].to_dict()

known_originals = set(original_labels.unique())
known_families  = {f"{fam}{FAMILY_LABEL_SUFFIX}" for fam in CELLTYPE_FAMILIES}
unknown = set(labels_df['label']) - known_originals - known_families - {AMBIGUOUS_LABEL}
if unknown:
    print(f"⚠ Labels not matching an original cell type or family: {sorted(unknown)}")

ct_to_family = {ct: fam for fam, cts in CELLTYPE_FAMILIES.items() for ct in cts}

def label_kind(lbl):
    if lbl is None:                       return 'no_label'
    if lbl == AMBIGUOUS_LABEL:            return 'ambiguous'
    if lbl.endswith(FAMILY_LABEL_SUFFIX): return 'family'
    return 'specific'

print(f"Loaded {len(labels_df)} (k, gep) labels across K = {sorted(labels_df['k'].unique())}")
print(labels_df['label'].map(label_kind).value_counts().to_string())


Loaded 204 (k, gep) labels across K = [np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20)]
label
ambiguous    113
family        50
specific      41


### 6. Baseline ROGUE — original labels

- ROGUE per (sample, original label), on the same HVG matrix everything else uses.
- Reference for `specific`-labeled GEPs in the split/merge signal (§9).


In [28]:
baseline_specific_path = os.path.join(OUT_DIR, "rogue_baseline_specific.csv")
if os.path.exists(baseline_specific_path):
    baseline_specific = pd.read_csv(baseline_specific_path)
else:
    baseline_specific = rogue_for_groups(original_labels, samples, desc="Baseline (specific)")
    baseline_specific = baseline_specific.rename(columns={'cluster': 'original_label'})
    baseline_specific.to_csv(baseline_specific_path, index=False)

baseline_specific_lookup = baseline_specific.set_index(['sample', 'original_label'])['ROGUE'].to_dict()
print(baseline_specific.groupby('original_label')['ROGUE'].median().round(3).to_string())


Baseline (specific):   0%|          | 0/312 [00:00<?, ?it/s]

original_label
DC_pDC             0.960
Mac_CXCL9          0.880
Mac_IL1B           0.868
Mac_IL1Bint        0.857
Mac_SPP1           0.890
Mast               0.975
Mono_CD14          0.982
Mono_CD14_ID1      0.943
Mono_CD14_IL1B     0.977
Mono_CD14_THBS1    0.981
Mono_CD16          0.970
Mono_Int           0.974
Mono_TIL           0.898
cDC1_CLEC9A        0.963
cDC2_CD1C          0.883
cDC2_CD33          0.962
mregDC_LAMP3       0.927


### 6b. Baseline ROGUE — family pools

- Maps each cell's original label to `<family>_family`, pools by (sample, family), scores ROGUE.
- Reference for `family`-labeled GEPs in the split/merge signal (§9).


In [29]:
baseline_family_path = os.path.join(OUT_DIR, "rogue_baseline_family.csv")

family_labels_per_cell = original_labels.map(
    lambda ct: f"{ct_to_family[ct]}{FAMILY_LABEL_SUFFIX}" if ct in ct_to_family else None
).dropna()

if os.path.exists(baseline_family_path):
    baseline_family = pd.read_csv(baseline_family_path)
else:
    baseline_family = rogue_for_groups(
        family_labels_per_cell, samples.loc[family_labels_per_cell.index],
        desc="Baseline (family)")
    baseline_family = baseline_family.rename(columns={'cluster': 'family_label'})
    baseline_family.to_csv(baseline_family_path, index=False)

baseline_family_lookup = baseline_family.set_index(['sample', 'family_label'])['ROGUE'].to_dict()
print(baseline_family.groupby('family_label')['ROGUE'].median().round(3).to_string())


Baseline (family):   0%|          | 0/132 [00:00<?, ?it/s]

family_label
DC_family            0.944
Macrophage_family    0.757
Monocyte_family      0.929


### 7. Whole-GEP ROGUE per (sample, K, GEP)

- For each K: hard-assign every cell to its argmax GEP.
- Drop cells whose GEP has no NB2 label; optionally drop `Ambiguous/Mixed` GEPs (`DROP_AMBIGUOUS`).
- Score ROGUE per (sample, GEP), within-sample so batch effects aren't read as impurity.
- `gep_ids_by_k` (per-cell GEP assignment) stays in memory — §8 reuses it directly instead of re-reading usage matrices.


In [30]:
usage_pattern = os.path.join(
    CNMF_DIR, f"{CNMF_PREFIX}.usages.k_*.dt_{DENSITY_THRESHOLD}.consensus.txt"
)
k_values = sorted({
    int(m.group(1))
    for f in glob.glob(usage_pattern)
    if (m := re.search(r'k_(\d+)', os.path.basename(f))) and K_MIN <= int(m.group(1)) <= K_MAX
})
k_values = [k for k in k_values if k in set(labels_df['k'].unique())]
print(f"K values: {k_values}")

def load_argmax(k):
    usage_path = os.path.join(
        CNMF_DIR, f"{CNMF_PREFIX}.usages.k_{k}.dt_{DENSITY_THRESHOLD}.consensus.txt"
    )
    u = pd.read_csv(usage_path, sep='\t', index_col=0)
    u.index, u.columns = u.index.astype(str), u.columns.astype(str)
    u = u.loc[u.index.intersection(counts_df_hvg.columns)]
    return u.idxmax(axis=1).astype(str).rename('gep')

gep_ids_by_k = {k: load_argmax(k) for k in k_values}

def _gep_out_path(k):
    return os.path.join(OUT_DIR, f"rogue_per_gep_K{k}.csv")  # same schema/name as before, so existing cache is reused

gep_rogue_parts = []
for k in k_values:
    out_path = _gep_out_path(k)
    if os.path.exists(out_path):
        gep_rogue_parts.append(pd.read_csv(out_path, dtype={'gep': str, 'label': str}))
        continue

    gep_ids = gep_ids_by_k[k]
    gep_ids = gep_ids[gep_ids.map(lambda g: (k, g) in label_lookup)]
    if DROP_AMBIGUOUS:
        keep = {g for g in gep_ids.unique() if label_lookup[(k, g)] != AMBIGUOUS_LABEL}
        gep_ids = gep_ids[gep_ids.isin(keep)]

    long = rogue_for_groups(gep_ids, samples.loc[gep_ids.index], desc=f"K={k}")
    long = long.rename(columns={'cluster': 'gep'})
    long['K'] = k
    long['label'] = long['gep'].map(lambda g: label_lookup.get((k, g)))
    long['label_kind'] = long['label'].map(label_kind)
    long = long[['sample', 'K', 'gep', 'label', 'label_kind', 'ROGUE', 'n_cells']]
    long.to_csv(out_path, index=False)
    gep_rogue_parts.append(long)

gep_rogue = pd.concat(gep_rogue_parts, ignore_index=True)
print(f"{len(gep_rogue)} (sample, K, GEP) rows")
print(gep_rogue['label_kind'].value_counts().to_string())


K values: [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


K=4:   0%|          | 0/126 [00:00<?, ?it/s]

K=5:   0%|          | 0/133 [00:00<?, ?it/s]

K=6:   0%|          | 0/148 [00:00<?, ?it/s]

K=7:   0%|          | 0/169 [00:00<?, ?it/s]

...

K=17:   0%|          | 0/108 [00:00<?, ?it/s]

K=18:   0%|          | 0/106 [00:00<?, ?it/s]

K=19:   0%|          | 0/260 [00:00<?, ?it/s]

K=20:   0%|          | 0/255 [00:00<?, ?it/s]

3071 (sample, K, GEP) rows
label_kind
ambiguous    1099
specific     1095
family        877


### 8. Comparison 1 — Contamination diagnostic

- For each GEP, take only the cells whose *original* label matches the GEP's NB2 label
  (exact match for `specific`, any family member for `family`).
- Score that label-matched subset and compare it to the whole GEP's ROGUE.
- A subset restricted to one label is expected to be at least as pure as the mixed GEP it
  came from — that's structural, not evidence of a real subtype.
- Read this as **contamination**, not a split/merge signal: a large gap means the GEP pulled
  in a lot of cells the original annotation didn't call this label.


In [31]:
def matched_cells(sample, k, gep, label, kind):
    cells = gep_ids_by_k[k].index[gep_ids_by_k[k] == gep]
    cells = cells.intersection(samples.index[samples == sample])
    if kind == 'specific':
        target = {label}
    elif kind == 'family':
        target = set(CELLTYPE_FAMILIES.get(label[:-len(FAMILY_LABEL_SUFFIX)], []))
    else:
        return cells[:0]
    return cells[original_labels.loc[cells].isin(target)]

contam_path = os.path.join(OUT_DIR, "rogue_contamination.csv")
if os.path.exists(contam_path):
    contamination = pd.read_csv(contam_path)
else:
    scoreable = gep_rogue[gep_rogue['label_kind'].isin(['specific', 'family'])]
    rows = []
    for r in tqdm(list(scoreable.itertuples(index=False)), desc="Contamination"):
        cells = matched_cells(r.sample, r.K, r.gep, r.label, r.label_kind)
        n_matched = len(cells)
        subset_rogue = rogue_for_cells(cells) if n_matched >= MIN_CELLS_PER_GROUP else np.nan
        rows.append({
            'sample': r.sample, 'K': r.K, 'gep': r.gep, 'label': r.label,
            'label_kind': r.label_kind, 'whole_gep_ROGUE': r.ROGUE, 'whole_gep_n_cells': r.n_cells,
            'matched_n_cells': n_matched, 'matched_frac': n_matched / r.n_cells,
            'matched_ROGUE': subset_rogue,
            'contamination_delta': subset_rogue - r.ROGUE if not pd.isna(subset_rogue) else np.nan,
        })
    contamination = pd.DataFrame(rows)
    contamination.to_csv(contam_path, index=False)

print(f"{len(contamination)} rows scored")
print(contamination[['matched_frac', 'contamination_delta']].describe().round(3).to_string())


Contamination:   0%|          | 0/1972 [00:00<?, ?it/s]

1972 rows scored
       matched_frac  contamination_delta
count      1972.000             1581.000
mean          0.714                0.016
std           0.356                0.028
min           0.000               -0.043
25%           0.484                0.000
50%           0.895                0.001
75%           1.000                0.025
max           1.000                0.216


### 9. Comparison 2 — Split/merge signal

- Whole-GEP ROGUE vs. the original-annotation baseline for that label:
  `baseline_specific` for `specific` labels, `baseline_family` for `family` labels.
- This is the legitimate "did cNMF's grouping beat the original annotation" question — both
  sides are ROGUE on the same HVG matrix, so the delta is comparable.
- `ambiguous` / `no_label` GEPs have no baseline and are reported as such, not scored.


In [32]:
def split_merge_baseline(row):
    if row['label_kind'] == 'specific':
        return baseline_specific_lookup.get((row['sample'], row['label']), np.nan)
    if row['label_kind'] == 'family':
        return baseline_family_lookup.get((row['sample'], row['label']), np.nan)
    return np.nan

def tag_delta(delta):
    if pd.isna(delta):                  return 'no_baseline'
    if delta >=  DELTA_ROGUE_THRESHOLD: return 'purer'
    if delta <= -DELTA_ROGUE_THRESHOLD: return 'less_pure'
    return 'tie'

split_merge = gep_rogue.copy()
split_merge['baseline_ROGUE'] = split_merge.apply(split_merge_baseline, axis=1)
split_merge['delta_ROGUE'] = split_merge['ROGUE'] - split_merge['baseline_ROGUE']
split_merge.loc[split_merge['label_kind'] == 'ambiguous', 'delta_ROGUE'] = np.nan
split_merge['recommendation'] = split_merge['delta_ROGUE'].map(tag_delta)
split_merge.loc[split_merge['label_kind'].isin(['ambiguous', 'no_label']), 'recommendation'] = split_merge['label_kind']

out = os.path.join(OUT_DIR, "rogue_split_merge_signal.csv")
split_merge.to_csv(out, index=False)
print(split_merge['recommendation'].value_counts().to_string())


recommendation
ambiguous      1099
tie            1005
purer           461
no_baseline     376
less_pure       130


### 10. Comparison 3 — Null / chance control

- cNMF defines GEPs *and* ROGUE scores them in the same HVG space — a GEP can look pure just
  because it was built to be internally coherent in that space, independent of biology.
- Null: for each (sample, group-size bin), draw `N_NULL_DRAWS` random same-size cell groups
  from that sample's full pool and score them the same way.
- A GEP only counts as evidence of real structure if it beats this null — beating the
  original-label baseline (§9) alone isn't enough, since that baseline says nothing about
  what random groups in this same gene space would already look like.


In [33]:
rng = np.random.default_rng(RANDOM_SEED)

def size_bin(n, width=NULL_SIZE_BIN):
    return max(width, int(round(n / width) * width))

gep_rogue['size_bin'] = gep_rogue['n_cells'].map(size_bin)
null_needed = gep_rogue[['sample', 'size_bin']].drop_duplicates()

null_path = os.path.join(OUT_DIR, "rogue_null_draws.csv")
if os.path.exists(null_path):
    null_draws = pd.read_csv(null_path)
else:
    rows = []
    for r in tqdm(list(null_needed.itertuples(index=False)), desc="Null draws"):
        pool = np.array(samples.index[samples == r.sample].intersection(counts_df_hvg.columns))
        if len(pool) < r.size_bin:
            continue
        for draw in range(N_NULL_DRAWS):
            cells = rng.choice(pool, size=r.size_bin, replace=False)
            rows.append({'sample': r.sample, 'size_bin': r.size_bin, 'draw': draw,
                        'ROGUE': rogue_for_cells(cells)})
    null_draws = pd.DataFrame(rows).dropna(subset=['ROGUE'])
    null_draws.to_csv(null_path, index=False)

null_summary = (null_draws.groupby(['sample', 'size_bin'])['ROGUE']
                .agg(null_median='median', null_mean='mean', null_std='std', n_draws='count')
                .reset_index())

null_control = gep_rogue.merge(null_summary, on=['sample', 'size_bin'], how='left')
null_control['null_zscore'] = (
    (null_control['ROGUE'] - null_control['null_mean']) / null_control['null_std']
)
null_control['beats_null_median'] = null_control['ROGUE'] > null_control['null_median']

out = os.path.join(OUT_DIR, "rogue_null_control.csv")
null_control.to_csv(out, index=False)
print(f"Beats null median: {null_control['beats_null_median'].mean():.1%} of rows")
print(null_control.groupby('label_kind')['beats_null_median'].mean().round(3).to_string())


Null draws:   0%|          | 0/532 [00:00<?, ?it/s]

Beats null median: 71.9% of rows
label_kind
ambiguous    0.622
family       0.742
specific     0.797
